In [4]:
import pandas as pd
import numpy as np
import ast
import random
import os

# Set random seed for reproducible bootstrapping
random.seed(42)
np.random.seed(42)

# File paths
input_files = [
    "/global/scratch/projects/fc_mvslab/OpenProjects/Claire/consensus_predictor/predictions/combined_preds/non_cl_seqs_all_six_combined_filtered.csv",
    "/global/scratch/projects/fc_mvslab/OpenProjects/Claire/consensus_predictor/predictions/combined_preds/y1000_filtered_all_six_combined.csv"
]
output_dir = "/global/scratch/projects/fc_mvslab/OpenProjects/Marissa/SynADs/2026-09-14_ComparePredictionsOnCADNNTrainingData"

os.makedirs(output_dir, exist_ok=True)

# Model definitions and thresholds
# FIXED: adhunter_ensemble window changed to 53
MODELS = {
    'adhunter': {'col': 'adhunter_preds', 'window': 40, 'thresholds': [100000, 150000]},
    'adhunter_ensemble': {'col': 'adhunter_ensemble_preds', 'window': 53, 'thresholds': [0, 0.44]},
    'tada': {'col': 'tada_preds', 'window': 40, 'thresholds': [0.4, 0.6]},
    'paddle': {'col': 'paddle_preds', 'window': 53, 'thresholds': [4, 6]},
    'adpred_10core': {'col': 'adpred_preds', 'window': 1, 'thresholds': [0.8]},
    'adpred_40core': {'col': 'adpred_preds', 'window': 1, 'thresholds': [0.8]}
}

BOOTSTRAP_ITERATIONS = 3
CHUNK_SIZE = 25000

In [5]:
def get_contiguous_blocks(mask):
    """Find start and end indices of contiguous True blocks in a boolean mask."""
    padded = np.concatenate(([False], mask, [False]))
    diff = np.diff(padded.astype(int))
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]
    return list(zip(starts, ends))

def calc_composition(seq_slice, scale_53=False):
    """Calculate compositional metrics for a sequence slice."""
    metrics = {aa: seq_slice.count(aa) for aa in ['W', 'F', 'Y', 'L', 'D', 'E', 'R', 'K']}
    metrics['Num_WFYL'] = metrics['W'] + metrics['F'] + metrics['Y'] + metrics['L']
    metrics['Num_WFYLM'] = metrics['Num_WFYL'] + seq_slice.count('M')
    metrics['Net_Charge'] = (metrics['R'] + metrics['K']) - (metrics['D'] + metrics['E'])
    
    # FIXED: Renamed argument to dynamically scale any 53-mer predictor
    if scale_53:
        scale_factor = 40.0 / 53.0
        for k in metrics:
            metrics[k] = metrics[k] * scale_factor
            
    return metrics

def validate_and_parse(pred_str, seq_len, window_size, model_name):
    """Parse string list to float array and strictly validate dimensions."""
    preds = np.array(ast.literal_eval(pred_str), dtype=float)
    expected_len = seq_len - window_size + 1
    
    if len(preds) != expected_len:
        raise ValueError(
            f"Dimension mismatch for {model_name}. Sequence length: {seq_len}, "
            f"Window: {window_size}. Expected {expected_len} predictions, got {len(preds)}."
        )
    return preds

def clamp_to_40(idx, seq_len):
    """Return a 40-residue slice centered around idx, clamping to edges."""
    start = idx - 19
    end = idx + 21
    
    if start < 0:
        start = 0
        end = 40
    elif end > seq_len:
        end = seq_len
        start = seq_len - 40
        
    return start, end

In [6]:
results = []
background_results = []

for file_path in input_files:
    print(f"Processing {os.path.basename(file_path)}...")
    chunk_iter = pd.read_csv(file_path, chunksize=CHUNK_SIZE)
    
    for chunk_idx, chunk in enumerate(chunk_iter):
        print(f"  Chunk {chunk_idx + 1}")
        
        for _, row in chunk.iterrows():
            seq = row['sequence']
            seq_len = len(seq)
            
            if seq_len < 40:
                continue
                
            # --- 1. Background Sampling ---
            for i in range(BOOTSTRAP_ITERATIONS):
                if seq_len == 40:
                    bg_start = 0
                else:
                    bg_start = random.randint(0, seq_len - 40)
                bg_seq = seq[bg_start : bg_start + 40]
                
                comp = calc_composition(bg_seq, scale_53=False)
                comp.update({
                    'Model': 'Background', 'Threshold': 'N/A', 
                    'Iteration': i, 'Length': 40
                })
                background_results.append(comp)

            # --- 2. Predictor Processing ---
            for model, params in MODELS.items():
                # FIXED: Skip sequence if it's too short for this specific model's window
                if seq_len < params['window']:
                    continue
                    
                col = params['col']
                if pd.isna(row[col]):
                    raise ValueError(f"Missing predictions in column {col} for sequence {row['name']}")
                
                preds = validate_and_parse(row[col], seq_len, params['window'], model)
                
                for thresh in params['thresholds']:
                    mask = preds > thresh
                    blocks = get_contiguous_blocks(mask)
                    
                    for start, end in blocks:
                        block_len = end - start
                        
                        if model == 'adpred_10core':
                            if block_len >= 10:
                                for i in range(BOOTSTRAP_ITERATIONS):
                                    core_idx = random.randint(start, end - 1)
                                    slice_start, slice_end = clamp_to_40(core_idx, seq_len)
                                    active_seq = seq[slice_start:slice_end]
                                    
                                    comp = calc_composition(active_seq, scale_53=False)
                                    comp.update({'Model': model, 'Threshold': thresh, 'Iteration': i})
                                    results.append(comp)
                                    
                        elif model == 'adpred_40core':
                            if block_len >= 40:
                                for i in range(BOOTSTRAP_ITERATIONS):
                                    slice_start = random.randint(start, end - 40)
                                    active_seq = seq[slice_start : slice_start + 40]
                                    
                                    comp = calc_composition(active_seq, scale_53=False)
                                    comp.update({'Model': model, 'Threshold': thresh, 'Iteration': i})
                                    results.append(comp)
                        
                        else:
                            for i in range(BOOTSTRAP_ITERATIONS):
                                passing_idx = random.randint(start, end - 1)
                                slice_start = passing_idx
                                slice_end = passing_idx + params['window']
                                active_seq = seq[slice_start:slice_end]
                                
                                # FIXED: dynamically scale based on window size
                                needs_scaling = (params['window'] == 53)
                                comp = calc_composition(active_seq, scale_53=needs_scaling)
                                
                                comp.update({'Model': model, 'Threshold': thresh, 'Iteration': i})
                                results.append(comp)

print("Processing complete.")

Processing non_cl_seqs_all_six_combined_filtered.csv...
  Chunk 1
  Chunk 2
  Chunk 3
  Chunk 4
  Chunk 5
  Chunk 6
Processing y1000_filtered_all_six_combined.csv...
  Chunk 1
  Chunk 2
  Chunk 3
  Chunk 4
Processing complete.


In [7]:
# Convert accumulated results to DataFrames
df_results = pd.DataFrame(results)
df_background = pd.DataFrame(background_results)

# Combine active and background distributions
df_combined = pd.concat([df_results, df_background], ignore_index=True)

# Save compressed results
output_file = os.path.join(output_dir, "compositional_biases_all_models_bootstrapped.csv")
df_combined.to_csv(output_file, index=False)

print(f"Saved {len(df_combined)} rows to {output_file}")

# Quick sanity check summary
summary = df_combined.groupby(['Model', 'Threshold'])['Net_Charge'].agg(['count', 'mean', 'median', 'min', 'max'])
display(summary)

Saved 12949857 rows to /global/scratch/projects/fc_mvslab/OpenProjects/Marissa/SynADs/2026-09-14_ComparePredictionsOnCADNNTrainingData/compositional_biases_all_models_bootstrapped.csv


count      mean    median        min        max
Model             Threshold                                                   
Background        N/A         690867  0.155422  0.000000 -40.000000  38.000000
adhunter          100000.0   2176095 -3.794614 -4.000000 -34.000000  14.000000
                  150000.0    645918 -5.663652 -6.000000 -27.000000   6.000000
adhunter_ensemble 0.0        4265895 -2.022281 -2.264151 -35.471698  11.320755
                  0.44       1137120 -4.145935 -3.773585 -18.867925   9.056604
adpred_10core     0.8         536583 -5.192265 -5.000000 -40.000000   6.000000
adpred_40core     0.8          13878 -8.743911 -8.000000 -40.000000   0.000000
paddle            4.0         556476 -5.100658 -4.528302 -24.150943   6.037736
                  6.0         319800 -5.943776 -6.037736 -20.377358   4.528302
tada              0.4        2060688 -4.268738 -4.000000 -27.000000   7.000000
                  0.6         546537 -6.675074 -6.000000 -23.000000   2.000000